# 04. Meta Prophet Model Training & Hyperparameter Tuning
### Energy Demand Forecasting Pipeline

This notebook tunes and trains the primary Bayesian generalized additive model (Meta Prophet) with German national holiday calendars.

**Key Operations:**
- Expanding-window rolling cross-validation across 3 folds to prevent temporal data leakage
- Grid search optimization over changepoint prior scale and seasonality parameters
- Model training with 95% Bayesian uncertainty prediction intervals
- Monthly seasonality component (period=30.5 days, Fourier order=5)
- Full success metrics: MAE, RMSE, MAPE, Forecast Accuracy, R², 95% Coverage, F1, Precision, Recall
- Export trained model to JSON (`ml/artifacts/models/prophet-daily-v1.json`)

## 1. Setup & Load Processed Splits

In [ ]:
import os
import sys
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress Prophet/Stan noise
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

try:
    from IPython.display import display
except ImportError:
    display = print

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("energy_forecasting")

# Robust directory discovery
for base in [Path("."), Path(".."), Path("../..")]:
    candidate = base / "ml" / "data"
    if candidate.exists():
        PROJECT_ROOT = base.resolve()
        break
else:
    PROJECT_ROOT = Path(".").resolve()

ML_DIR = PROJECT_ROOT / "ml"
DATA_DIR = ML_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
ARTIFACTS_DIR = ML_DIR / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
FORECASTS_DIR = ARTIFACTS_DIR / "forecasts"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

for d in [PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, FORECASTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DS_COL = "ds"
Y_COL = "y"

print(f"Project root  : {PROJECT_ROOT}")

train_daily = pd.read_csv(PROCESSED_DIR / "train_daily.csv", parse_dates=[DS_COL])
test_daily  = pd.read_csv(PROCESSED_DIR / "test_daily.csv",  parse_dates=[DS_COL])
print(f"Loaded train_daily ({len(train_daily):,} rows) and test_daily ({len(test_daily):,} rows).")
print(f"Train period : {train_daily[DS_COL].min().date()} → {train_daily[DS_COL].max().date()}")
print(f"Test period  : {test_daily[DS_COL].min().date()} → {test_daily[DS_COL].max().date()}")

## 2. Expanding-Window Cross-Validation & Grid Search

All tuning runs are performed **strictly inside the training partition** (first 70%).
The test set is **never accessed** during this phase — guaranteeing zero temporal data leakage.

Each fold trains on an expanding window and validates on the next `horizon`-day block.

In [ ]:
# Expanding-Window Cross-Validation & Hyperparameter Tuning
from prophet import Prophet

def run_expanding_window_cv(
    train_df: pd.DataFrame,
    param_grid: list,
    n_splits: int = 3,
    horizon: int = 90,
    add_monthly_seasonality: bool = True,
):
    """
    Evaluate hyperparameter combinations across expanding temporal windows.

    ZERO LEAKAGE GUARANTEE:
        - Only the training partition (first 70%) is passed to this function.
        - Validation folds are carved from within the training partition.
        - The holdout test set is NEVER visible here.

    Parameters
    ----------
    train_df               : Training DataFrame with [ds, y] columns (ONLY training partition).
    param_grid             : List of hyperparameter dicts to evaluate.
    n_splits               : Number of expanding-window folds.
    horizon                : Days ahead for each validation fold.
    add_monthly_seasonality: If True, adds intra-month Fourier seasonality (period=30.5, order=5).

    Returns
    -------
    pd.DataFrame sorted by CV_MAE (ascending). Also prints per-trial results.
    """
    results = []
    total_len = len(train_df)

    # Reserve the last (n_splits * horizon) rows for validation
    start_train_size = total_len - (n_splits * horizon)
    if start_train_size < horizon * 2:
        start_train_size = int(total_len * 0.6)
        horizon = int((total_len - start_train_size) / n_splits)

    print(f"{'─'*70}")
    print(f"[CV] Training rows   : {total_len:,}")
    print(f"[CV] Initial window  : {start_train_size:,} rows")
    print(f"[CV] Fold horizon    : {horizon} days | Folds: {n_splits}")
    print(f"[CV] Configs to try  : {len(param_grid)}")
    print(f"{'─'*70}")

    for p_idx, params in enumerate(param_grid):
        fold_maes = []
        fold_rmses = []
        for fold in range(n_splits):
            train_end = start_train_size + fold * horizon
            val_end   = train_end + horizon

            fold_train = train_df.iloc[:train_end].copy()
            fold_val   = train_df.iloc[train_end:val_end].copy()

            # Leakage sanity check
            assert fold_train[DS_COL].max() < fold_val[DS_COL].min(), (
                f"Leakage detected in fold {fold+1}!"
            )

            m = Prophet(**params)
            m.add_country_holidays(country_name='DE')
            if add_monthly_seasonality:
                m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
            m.fit(fold_train)

            future = fold_val[[DS_COL]].copy()
            fcst   = m.predict(future)

            errors    = fold_val[Y_COL].values - fcst['yhat'].values
            fold_mae  = np.mean(np.abs(errors))
            fold_rmse = np.sqrt(np.mean(errors ** 2))
            fold_maes.append(fold_mae)
            fold_rmses.append(fold_rmse)

        mean_mae  = np.mean(fold_maes)
        mean_rmse = np.mean(fold_rmses)
        results.append({"params": params, "CV_MAE": round(mean_mae, 2), "CV_RMSE": round(mean_rmse, 2)})
        print(f"  Trial [{p_idx+1}/{len(param_grid)}] cps={params['changepoint_prior_scale']:.2f} "
              f"sps={params['seasonality_prior_scale']:.1f} "
              f"hps={params.get('holidays_prior_scale', '—')} "
              f"→ CV MAE: {mean_mae:,.2f} MW | CV RMSE: {mean_rmse:,.2f} MW")

    results_df = pd.DataFrame(results).sort_values("CV_MAE").reset_index(drop=True)
    return results_df


# Expanded grid matching optimized pipeline configuration
param_grid = [
    {"changepoint_prior_scale": 0.01,  "seasonality_prior_scale": 1.0,  "holidays_prior_scale": 1.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 5.0,  "holidays_prior_scale": 10.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
    {"changepoint_prior_scale": 0.05,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
    {"changepoint_prior_scale": 0.10,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
    {"changepoint_prior_scale": 0.15,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
    {"changepoint_prior_scale": 0.20,  "seasonality_prior_scale": 10.0, "holidays_prior_scale": 10.0,
     "seasonality_mode": "additive", "yearly_seasonality": True, "weekly_seasonality": True, "daily_seasonality": False},
]

print("Running expanding-window cross-validation on daily training set...")
print("(Test set is completely isolated — zero data leakage guarantee)\n")
tuning_results = run_expanding_window_cv(train_daily, param_grid, n_splits=3, horizon=90)
best_params = tuning_results.iloc[0]["params"]
print(f"\n{'─'*70}")
print(f"Best Hyperparameters selected (lowest CV MAE):")
print(json.dumps({k: v for k, v in best_params.items()}, indent=2))
print(f"{'─'*70}")
display(tuning_results[["CV_MAE", "CV_RMSE", "params"]])

## 3. Production Model Training & Serialization

In [ ]:
# Train Production Meta Prophet Model with German Holiday Calendar
from prophet.serialize import model_to_json, model_from_json

# Configure production model with selected optimal parameters and 95% Bayesian intervals
prod_params = {
    **best_params,
    "holidays_prior_scale": best_params.get("holidays_prior_scale", 10.0),
    "interval_width": 0.95,
    "n_changepoints": 25,
    "changepoint_range": 0.8,
}

print(f"Training final Prophet model on {len(train_daily):,} daily observations...")
print(f"Parameters: {json.dumps({k: v for k, v in prod_params.items()}, indent=2)}")

daily_prophet_model = Prophet(**prod_params)
daily_prophet_model.add_country_holidays(country_name='DE')
daily_prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
daily_prophet_model.fit(train_daily)

# Generate forecast on full holdout test set
future_test    = test_daily[[DS_COL]].copy()
forecast_daily = daily_prophet_model.predict(future_test)

# Save model and predictions
model_save_path = MODELS_DIR / "prophet-daily-v1.json"
with open(model_save_path, "w") as f:
    f.write(model_to_json(daily_prophet_model))

forecast_save_path = FORECASTS_DIR / "final_predictions_daily.csv"
forecast_daily.to_csv(forecast_save_path, index=False)

print(f"\nModel saved    : {model_save_path}")
print(f"Forecasts saved: {forecast_save_path}")
forecast_daily[[DS_COL, 'yhat', 'yhat_lower', 'yhat_upper']].head(10)

## 4. Comprehensive Success Metrics

All metrics are computed on the **holdout test set** — first and only access to test data:

| Metric | Description |
|---|---|
| **MAE** | Mean Absolute Error (MW) |
| **RMSE** | Root Mean Squared Error (MW) |
| **MAPE** | Mean Absolute Percentage Error |
| **Forecast Accuracy** | 100 − MAPE |
| **R² Score** | Coefficient of Determination |
| **95% Coverage** | Fraction of actuals within prediction interval |
| **Anomaly F1 / Precision / Recall** | Classification metrics vs. 2.5σ statistical benchmark |

In [ ]:
y_true   = test_daily[Y_COL].values
y_pred   = forecast_daily['yhat'].values
y_lower  = forecast_daily['yhat_lower'].values
y_upper  = forecast_daily['yhat_upper'].values

# ── Regression metrics ─────────────────────────────────────────────────────
mae_val  = float(np.mean(np.abs(y_true - y_pred)))
rmse_val = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
mape_val = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1.0))) * 100)
acc_val  = float(np.clip(100.0 - mape_val, 0, 100))

ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2_val = float(1.0 - ss_res / ss_tot) if ss_tot != 0 else 0.0

inside   = (y_true >= y_lower) & (y_true <= y_upper)
cov_val  = float(np.mean(inside) * 100.0)

print(f"{'═'*60}")
print(f"  PROPHET FORECASTING METRICS (Daily, Holdout Test Set)")
print(f"{'═'*60}")
print(f"  MAE              : {mae_val:>10,.2f} MW")
print(f"  RMSE             : {rmse_val:>10,.2f} MW")
print(f"  MAPE             : {mape_val:>10.2f} %")
print(f"  Forecast Accuracy: {acc_val:>10.2f} %")
print(f"  R² Score         : {r2_val:>10.4f}")
print(f"  95% PI Coverage  : {cov_val:>10.2f} %")
print(f"{'═'*60}")

# ── Anomaly Detection Classification Metrics ───────────────────────────────
deviation = y_true - y_pred
dev_std   = float(np.std(deviation))
ref_anomaly   = np.abs(deviation) >= (2.5 * dev_std)
pred_anomaly  = (y_true < y_lower) | (y_true > y_upper)

y_t = np.where(ref_anomaly,  "ANOMALY", "NORMAL")
y_p = np.where(pred_anomaly, "ANOMALY", "NORMAL")

tp = int(np.sum((y_t == "ANOMALY") & (y_p == "ANOMALY")))
fp = int(np.sum((y_t == "NORMAL")  & (y_p == "ANOMALY")))
tn = int(np.sum((y_t == "NORMAL")  & (y_p == "NORMAL")))
fn = int(np.sum((y_t == "ANOMALY") & (y_p == "NORMAL")))

anomaly_acc  = float(np.mean(y_t == y_p) * 100)
precision    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall       = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1           = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n  ANOMALY DETECTION METRICS (vs. 2.5σ statistical benchmark)")
print(f"{'─'*60}")
print(f"  Reference anomalies (2.5σ) : {int(ref_anomaly.sum())} of {len(y_true)} days")
print(f"  Anomaly Classification Acc : {anomaly_acc:.2f}%")
print(f"  Precision                  : {precision:.4f}")
print(f"  Recall                     : {recall:.4f}")
print(f"  F1 Score                   : {f1:.4f}")
print(f"  Confusion Matrix: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print(f"{'═'*60}")

## 5. Forecast Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# ── Plot 1: Actual vs Forecast with uncertainty band ──────────────────────
ax = axes[0]
ax.plot(test_daily[DS_COL], y_true, color='#0f172a', linewidth=1.5, label='Actual Demand', zorder=3)
ax.plot(forecast_daily[DS_COL], y_pred, color='#0d9488', linewidth=2.0, label='Prophet Forecast', zorder=4)
ax.fill_between(forecast_daily[DS_COL], y_lower, y_upper,
                color='#14b8a6', alpha=0.20, label='95% Prediction Interval')
ax.set_title(f'Prophet Daily Forecast — Holdout Test Set\n'
             f'MAE: {mae_val:,.0f} MW | RMSE: {rmse_val:,.0f} MW | '
             f'Accuracy: {acc_val:.1f}% | R²: {r2_val:.4f} | Coverage: {cov_val:.1f}%',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Load (MW)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

# ── Plot 2: Forecast Errors ───────────────────────────────────────────────
ax2 = axes[1]
errors = y_true - y_pred
ax2.bar(test_daily[DS_COL], errors,
        color=np.where(errors > 0, '#ef4444', '#3b82f6'), alpha=0.6, width=1)
ax2.axhline(0, color='black', linewidth=1.0)
ax2.axhline(errors.mean(), color='orange', linewidth=1.5, linestyle='--',
            label=f'Mean Error: {errors.mean():+.0f} MW')
ax2.set_title('Forecast Residuals (Actual − Forecast)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Residual (MW)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4)

plt.tight_layout()
fig_path = PROJECT_ROOT / 'reports' / 'figures' / 'notebook_04_forecast.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

## 6. Save Metrics Artifact

In [ ]:
# Save comprehensive metrics to JSON artifact
metrics_out = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "model": "prophet-daily-v1",
    "best_params": {k: v for k, v in best_params.items()},
    "forecasting_metrics": {
        "MAE_MW": round(mae_val, 4),
        "RMSE_MW": round(rmse_val, 4),
        "MAPE_pct": round(mape_val, 4),
        "forecast_accuracy_pct": round(acc_val, 4),
        "R2_score": round(r2_val, 4),
        "coverage_95_pct": round(cov_val, 2),
    },
    "anomaly_detection_metrics": {
        "accuracy_pct": round(anomaly_acc, 2),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "confusion_matrix": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
    },
    "cv_tuning_summary": tuning_results[["CV_MAE", "CV_RMSE"]].to_dict(orient="records"),
}

metrics_path = METRICS_DIR / "notebook_04_metrics.json"
with open(metrics_path, "w") as fh:
    json.dump(metrics_out, fh, indent=2, default=str)

print(f"Metrics saved to: {metrics_path}")
print(json.dumps(metrics_out["forecasting_metrics"], indent=2))
print(json.dumps(metrics_out["anomaly_detection_metrics"], indent=2))